# Cybersecurity Threat Intelligence Research Agent

A multi-agent system for conducting deep cybersecurity threat intelligence research. Given a threat topic, CVE, malware family, or attack technique, the pipeline:

1. **Guardrail Agent** — Validates the query is a legitimate security research topic
2. **Planner Agent** — Decomposes the topic into targeted search strategies
3. **Search Agent** — Queries threat databases (NVD, MITRE ATT&CK, Semantic Scholar, Shodan advisories)
4. **Analyst Agent** — Synthesizes findings into a structured threat intelligence report
5. **Email Agent** — Optionally delivers the report via EmailJS

Designed for security analysts, incident responders, red team operators, and researchers.

## Setup

In [ ]:
import os
import json
import re
import time
import requests
import ipywidgets as widgets
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display, clear_output
from datetime import datetime

In [ ]:
load_dotenv(override=True)

openai_api_key    = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
groq_api_key      = os.getenv("GROQ_API_KEY")
emailjs_service   = os.getenv("EMAILJS_SERVICE_ID")
emailjs_template  = os.getenv("EMAILJS_TEMPLATE_ID")
emailjs_key       = os.getenv("EMAILJS_PUBLIC_KEY")

for name, key, n in [
    ("OpenAI",    openai_api_key,    8),
    ("Anthropic", anthropic_api_key, 7),
    ("Groq",      groq_api_key,      4),
    ("EmailJS",   emailjs_key,       4),
]:
    print(f"{name}: {'set — begins ' + key[:n] if key else 'not set'}")

## Client Initialization

In [ ]:
openai_client    = OpenAI(api_key=openai_api_key) if openai_api_key else None
anthropic_client = Anthropic(api_key=anthropic_api_key) if anthropic_api_key else None
groq_client      = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1") if groq_api_key else None

## Query Input

In [ ]:
query_input = widgets.Textarea(
    placeholder="Enter a threat topic, CVE ID, malware family, or attack technique...\n\nExamples:\n  - CVE-2024-3400 Palo Alto GlobalProtect RCE\n  - Lazarus Group supply chain attack TTPs\n  - Ransomware-as-a-Service LockBit 3.0 infrastructure\n  - MITRE T1190 Exploit Public-Facing Application",
    description="Query:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="750px", height="140px"),
)

depth_input = widgets.RadioButtons(
    options=[("Quick Brief (500–800 words)", "brief"), ("Standard Report (1000–1500 words)", "standard"), ("Deep Dive (2000+ words)", "deep")],
    value="standard",
    description="Depth:",
    style={"description_width": "initial"},
)

analyst_model_input = widgets.Dropdown(
    options=[
        ("GPT-4o-mini (OpenAI)", "openai:gpt-4o-mini"),
        ("Claude 3.7 Sonnet (Anthropic)", "anthropic:claude-3-7-sonnet-latest"),
        ("Llama 3.3 70B (Groq)", "groq:llama-3.3-70b-versatile"),
    ],
    value="anthropic:claude-3-7-sonnet-latest",
    description="Analyst model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

email_input = widgets.Text(
    placeholder="your@email.com (optional — requires EmailJS keys)",
    description="Deliver to:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)

display(query_input, depth_input, analyst_model_input, email_input)

## Agent Definitions

In [ ]:
def call_llm(prompt: str, system: str = "", model_str: str = "openai:gpt-4o-mini", max_tokens: int = 2000) -> str:
    provider, model_name = model_str.split(":", 1)

    if provider == "openai":
        msgs = []
        if system:
            msgs.append({"role": "system", "content": system})
        msgs.append({"role": "user", "content": prompt})
        resp = openai_client.chat.completions.create(model=model_name, messages=msgs, max_tokens=max_tokens)
        return resp.choices[0].message.content

    elif provider == "anthropic":
        resp = anthropic_client.messages.create(
            model=model_name,
            max_tokens=max_tokens,
            system=system or "You are a cybersecurity threat intelligence analyst.",
            messages=[{"role": "user", "content": prompt}],
        )
        return resp.content[0].text

    elif provider == "groq":
        msgs = []
        if system:
            msgs.append({"role": "system", "content": system})
        msgs.append({"role": "user", "content": prompt})
        resp = groq_client.chat.completions.create(model=model_name, messages=msgs, max_tokens=max_tokens)
        return resp.choices[0].message.content

    raise ValueError(f"Unknown provider: {provider}")

In [ ]:
GUARDRAIL_SYSTEM = """You are a strict cybersecurity query validator. Your only job is to decide whether a query
is a legitimate cybersecurity or threat intelligence research topic.

ACCEPT queries about:
- CVEs, vulnerabilities, exploits, and patches
- Malware families, ransomware, APT groups, threat actors
- Attack techniques (MITRE ATT&CK TTPs)
- Network security, intrusion detection, incident response
- Security tools, defensive measures, threat hunting
- Supply chain attacks, zero-days, phishing campaigns
- Cryptographic weaknesses and protocol vulnerabilities

REJECT queries that are:
- Requests to write malware, exploits, or attack scripts
- Targeting specific individuals, organizations, or infrastructure for harm
- Clearly off-topic (finance, medicine, cooking, general coding, etc.)
- Social engineering scripts intended for actual use

Respond ONLY with valid JSON:
{"allowed": true, "reason": "brief explanation"}
or
{"allowed": false, "reason": "brief explanation", "suggestion": "rephrased acceptable version if possible"}"""


def guardrail_agent(query: str) -> dict:
    print("[Guardrail Agent] Validating query...")
    raw = call_llm(query, system=GUARDRAIL_SYSTEM, model_str="openai:gpt-4o-mini", max_tokens=300)
    try:
        clean = re.sub(r"```json|```", "", raw).strip()
        return json.loads(clean)
    except json.JSONDecodeError:
        return {"allowed": True, "reason": "Validation parse error — proceeding with caution"}

In [ ]:
PLANNER_SYSTEM = """You are a senior threat intelligence analyst and research strategist.
Given a cybersecurity query, produce a structured research plan that maximizes coverage of:
- Technical vulnerability details
- Threat actor attribution and history
- Attack lifecycle and kill chain mapping
- Affected systems and exploitation evidence
- Defensive mitigations and detection rules

Respond ONLY with valid JSON (no markdown):
{
  "topic_summary": "one sentence description",
  "search_queries": ["query1", "query2", ...],
  "nvd_cve_ids": ["CVE-XXXX-XXXXX"] or [],
  "mitre_techniques": ["T1XXX"] or [],
  "key_angles": ["angle1", "angle2", ...],
  "report_sections": ["section1", "section2", ...]
}"""


def planner_agent(query: str) -> dict:
    print("[Planner Agent] Building research plan...")
    raw = call_llm(query, system=PLANNER_SYSTEM, model_str="openai:gpt-4o-mini", max_tokens=800)
    try:
        clean = re.sub(r"```json|```", "", raw).strip()
        return json.loads(clean)
    except json.JSONDecodeError:
        return {
            "topic_summary": query,
            "search_queries": [query],
            "nvd_cve_ids": [],
            "mitre_techniques": [],
            "key_angles": ["technical analysis", "mitigations"],
            "report_sections": ["Overview", "Technical Details", "Mitigations"],
        }

In [ ]:
def fetch_nvd_cve(cve_id: str) -> dict:
    url = f"https://services.nvd.nist.gov/rest/json/cves/2.0?cveId={cve_id}"
    try:
        resp = requests.get(url, timeout=10, headers={"User-Agent": "ThreatIntelAgent/1.0"})
        if resp.status_code == 200:
            data = resp.json()
            vulns = data.get("vulnerabilities", [])
            if vulns:
                cve = vulns[0]["cve"]
                desc = next(
                    (d["value"] for d in cve.get("descriptions", []) if d["lang"] == "en"),
                    "No description available",
                )
                metrics = cve.get("metrics", {})
                cvss_score = None
                cvss_vector = None
                for version in ["cvssMetricV31", "cvssMetricV30", "cvssMetricV2"]:
                    if version in metrics and metrics[version]:
                        cvss_data = metrics[version][0].get("cvssData", {})
                        cvss_score = cvss_data.get("baseScore")
                        cvss_vector = cvss_data.get("vectorString")
                        break
                return {
                    "source": "NVD",
                    "cve_id": cve_id,
                    "description": desc,
                    "cvss_score": cvss_score,
                    "cvss_vector": cvss_vector,
                    "published": cve.get("published", ""),
                    "references": [r["url"] for r in cve.get("references", [])[:5]],
                }
    except Exception as e:
        return {"source": "NVD", "cve_id": cve_id, "error": str(e)}
    return {"source": "NVD", "cve_id": cve_id, "error": "Not found"}


def fetch_semantic_scholar(query: str, limit: int = 5) -> list:
    url = "https://api.semanticscholar.org/graph/v1/paper/search"
    params = {
        "query": query,
        "limit": limit,
        "fields": "title,abstract,year,authors,citationCount,externalIds,url",
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code == 200:
            papers = resp.json().get("data", [])
            results = []
            for p in papers:
                results.append({
                    "source": "Semantic Scholar",
                    "title": p.get("title", ""),
                    "abstract": (p.get("abstract") or "")[:600],
                    "year": p.get("year"),
                    "authors": ", ".join(a["name"] for a in p.get("authors", [])[:3]),
                    "citations": p.get("citationCount", 0),
                    "url": p.get("url", ""),
                })
            return results
    except Exception as e:
        return [{"source": "Semantic Scholar", "error": str(e)}]
    return []


def fetch_mitre_technique(technique_id: str) -> dict:
    url = f"https://attack.mitre.org/techniques/{technique_id}/"
    return {
        "source": "MITRE ATT&CK",
        "technique_id": technique_id,
        "url": url,
        "note": "See URL for full TTP details, sub-techniques, and detection guidance",
    }


def search_agent(plan: dict) -> dict:
    print("[Search Agent] Querying threat databases...")
    results = {"nvd": [], "papers": [], "mitre": [], "raw_query_results": []}

    for cve_id in plan.get("nvd_cve_ids", []):
        print(f"  Fetching NVD: {cve_id}")
        results["nvd"].append(fetch_nvd_cve(cve_id))
        time.sleep(0.6)

    for technique in plan.get("mitre_techniques", []):
        print(f"  Mapping MITRE: {technique}")
        results["mitre"].append(fetch_mitre_technique(technique))

    for query in plan.get("search_queries", [])[:4]:
        print(f"  Searching papers: '{query[:60]}'")
        papers = fetch_semantic_scholar(query, limit=4)
        results["papers"].extend(papers)
        time.sleep(0.5)

    seen = set()
    deduped = []
    for p in results["papers"]:
        title = p.get("title", "")
        if title and title not in seen:
            seen.add(title)
            deduped.append(p)
    results["papers"] = deduped[:12]

    print(f"  Retrieved: {len(results['nvd'])} CVEs, {len(results['papers'])} papers, {len(results['mitre'])} MITRE techniques")
    return results

In [ ]:
ANALYST_SYSTEM = """You are a senior cybersecurity threat intelligence analyst writing a formal threat intelligence report.
Your reports are used by SOC teams, incident responders, and executive stakeholders.

Writing standards:
- Use precise technical language but explain acronyms on first use
- Cite all sources by title or ID (NVD, Semantic Scholar, MITRE)
- Distinguish confirmed findings from assessments (use 'assessed with high/medium confidence')
- Map attack techniques to MITRE ATT&CK where possible
- Include CVSS scores and severity ratings where available
- End every report with a structured Recommendations section
- Use TLP:WHITE classification unless instructed otherwise"""


def analyst_agent(query: str, plan: dict, search_results: dict, depth: str, model_str: str) -> str:
    print(f"[Analyst Agent] Synthesizing report with {model_str.split(':')[1]}...")

    depth_instructions = {
        "brief": "Write a concise threat brief of 500–800 words. Focus on critical facts and top 3 recommendations.",
        "standard": "Write a comprehensive report of 1000–1500 words covering all key sections.",
        "deep": "Write an in-depth threat intelligence report of 2000+ words with full technical analysis, IOC discussion, and detection engineering guidance.",
    }

    nvd_block = json.dumps(search_results["nvd"], indent=2) if search_results["nvd"] else "No CVE data retrieved."
    papers_block = json.dumps(search_results["papers"], indent=2) if search_results["papers"] else "No academic papers retrieved."
    mitre_block = json.dumps(search_results["mitre"], indent=2) if search_results["mitre"] else "No MITRE techniques mapped."

    max_tokens = {"brief": 1200, "standard": 2200, "deep": 4000}[depth]

    prompt = f"""Research Query: {query}

Research Plan Summary:
- Topic: {plan.get('topic_summary', query)}
- Key Angles: {', '.join(plan.get('key_angles', []))}
- Requested Sections: {', '.join(plan.get('report_sections', []))}

CVE / Vulnerability Data:
{nvd_block}

Academic & Research Papers:
{papers_block}

MITRE ATT&CK Techniques:
{mitre_block}

Instructions:
{depth_instructions[depth]}

Structure the report with:
- TLP Classification and Date
- Executive Summary
- Technical Analysis
- Threat Actor / Attribution (if applicable)
- Attack Lifecycle (MITRE ATT&CK mapping where relevant)
- Affected Systems and Impact
- Indicators of Compromise (IOCs) — describe types even if specific values unavailable
- Detection and Hunting Guidance
- Recommendations (prioritized: Immediate / Short-term / Long-term)
- References

Use Markdown formatting."""

    return call_llm(prompt, system=ANALYST_SYSTEM, model_str=model_str, max_tokens=max_tokens)

In [ ]:
def email_agent(report: str, query: str, recipient_email: str) -> bool:
    if not all([emailjs_service, emailjs_template, emailjs_key]):
        print("[Email Agent] EmailJS keys not configured. Skipping delivery.")
        return False

    print(f"[Email Agent] Sending report to {recipient_email}...")
    payload = {
        "service_id": emailjs_service,
        "template_id": emailjs_template,
        "user_id": emailjs_key,
        "template_params": {
            "to_email": recipient_email,
            "subject": f"Threat Intelligence Report: {query[:80]}",
            "message": report[:8000],
            "date": datetime.now().strftime("%Y-%m-%d %H:%M UTC"),
        },
    }
    try:
        resp = requests.post("https://api.emailjs.com/api/v1.0/email/send", json=payload, timeout=10)
        if resp.status_code == 200:
            print("[Email Agent] Delivered successfully.")
            return True
        print(f"[Email Agent] Delivery failed: {resp.status_code} {resp.text}")
    except Exception as e:
        print(f"[Email Agent] Error: {e}")
    return False

## Run the Pipeline

In [ ]:
query         = query_input.value.strip()
depth         = depth_input.value
analyst_model = analyst_model_input.value
recipient     = email_input.value.strip()

if not query:
    raise ValueError("No query entered. Fill in the Query field above and re-run.")

print(f"Query    : {query}")
print(f"Depth    : {depth}")
print(f"Model    : {analyst_model}")
print(f"Delivery : {recipient or 'none'}")
print()

In [ ]:
guardrail_result = guardrail_agent(query)

if not guardrail_result.get("allowed", True):
    display(Markdown(
        f"**Query rejected by Guardrail Agent**\n\n"
        f"**Reason:** {guardrail_result.get('reason', 'Not a cybersecurity research topic.')}\n\n"
        + (
            f"**Try instead:** {guardrail_result['suggestion']}"
            if guardrail_result.get('suggestion') else ""
        )
    ))
    raise SystemExit("Pipeline halted by guardrail.")

print(f"Guardrail passed: {guardrail_result.get('reason', 'OK')}")

In [ ]:
plan = planner_agent(query)

display(Markdown(
    f"**Research Plan**\n\n"
    f"- Topic: {plan.get('topic_summary', '')}\n"
    f"- Search queries: {len(plan.get('search_queries', []))}\n"
    f"- CVEs to fetch: {plan.get('nvd_cve_ids', [])}\n"
    f"- MITRE techniques: {plan.get('mitre_techniques', [])}\n"
    f"- Report sections: {', '.join(plan.get('report_sections', []))}"
))

In [ ]:
search_results = search_agent(plan)

In [ ]:
report = analyst_agent(query, plan, search_results, depth, analyst_model)

display(Markdown("---\n## Threat Intelligence Report\n---\n\n" + report))

## Save and Deliver

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
safe_query = re.sub(r"[^\w\s-]", "", query[:40]).strip().replace(" ", "_")
output_path = f"threat_intel_{safe_query}_{timestamp}.md"

with open(output_path, "w") as f:
    f.write(f"# Threat Intelligence Report\n")
    f.write(f"**Query:** {query}\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M UTC')}\n")
    f.write(f"**Model:** {analyst_model}\n")
    f.write(f"**Depth:** {depth}\n\n---\n\n")
    f.write(report)

print(f"Report saved: {output_path}")

if recipient:
    email_agent(report, query, recipient)

## Raw Search Data (Optional Inspection)

In [ ]:
print(json.dumps(search_results, indent=2))